In [7]:
from datetime import datetime
import MetaTrader5 as mt5
import pandas as pd
import pytz
import time
import pandas_ta as pta
import matplotlib.pyplot as plt 
# import threading
mt5.initialize()


True

In [8]:
def Action(symbol, lot, signal):
    try:
        symbol_info = mt5.symbol_info(symbol)

        a = [[mt5.ORDER_TYPE_SELL, mt5.symbol_info_tick(symbol).bid], [mt5.ORDER_TYPE_BUY, mt5.symbol_info_tick(symbol).ask]]
        price = a[signal][1]
        deviation = 200
        
        request = {
            "action": mt5.TRADE_ACTION_DEAL,
            "symbol": symbol,
            "volume": lot,
            "type": a[signal][0],
            "price": price,
            "deviation": deviation,
            "magic": 234000,
            "comment": "python script open",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_FOK,
        }
        result = mt5.order_send(request)
        return result
    except Exception as e:
        print("Action")
        print(e)

In [9]:
def price_action(symbol, lot, ask, bid, order_type):
    buy_profit=mt5.order_calc_profit(order_type,symbol,lot,ask,bid)
    return buy_profit
price_action("GBPJPY", 1.0, 1.18969, 1.17030,mt5.ORDER_TYPE_SELL)

12.98

In [10]:
def get_rsi(close, lookback):
#     t = time.time()
    ret = close.diff()
    
    up = []
    down = []
    for i in range(len(ret)):
        if ret[i] < 0:
            up.append(0)
            down.append(ret[i])
        else:
            up.append(ret[i])
            down.append(0)
    up_series = pd.Series(up)
    down_series = pd.Series(down).abs()
    up_ewm = up_series.ewm(com = lookback - 1, adjust = False).mean()
    down_ewm = down_series.ewm(com = lookback - 1, adjust = False).mean()
    rs = up_ewm/down_ewm
    rsi = 100 - (100 / (1 + rs))
    rsi_df = pd.DataFrame(rsi).rename(columns = {0:'rsi'}).set_index(close.index)
#     print(time.time()-t)
    return rsi_df

In [11]:
def ema(s, n):
    ema = []
    j = 1

    #get n sma first and calculate the next n period ema
    sma = sum(s[:n]) / n
    multiplier = 2 / float(1 + n)
    ema.append(sma)

    #EMA(current) = ( (Price(current) - EMA(prev) ) x Multiplier) + EMA(prev)
    ema.append(( (s[n] - sma) * multiplier) + sma)

    #now calculate the rest of the values
    for i in s[n+1:]:
        tmp = ( (i - ema[j]) * multiplier) + ema[j]
        j = j + 1
        ema.append(tmp)

    return ema

In [4]:
def get_values(symbol, size):
    rates = mt5.copy_rates_from_pos(symbol, mt5.TIMEFRAME_H3, 0, size)

    rates_frame = pd.DataFrame(rates)
    
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume',], axis=1)
    fastma = ema(rates_frame['close'], 12)
    slowma = ema(rates_frame['close'], 26)
#     signalLength = ema(rates_frame['close'], 9)
    
    rates_frame['fastma'] = fastma
    rates_frame['slowma'] = slowma
    rates_frame['macd'] = rates_frame['fastma'] - rates_frame['slowma']
    rates_frame['signal'] = rates_frame['macd'].rolling(window=9).mean()
    rates_frame['hist'] = rates_frame['macd'] - rates_frame['signal']
    
    rates_frame['nissOscRaw'] = [0]*69 + list(map(lambda x,y : ((x - y)/y)*100, rates_frame['nissFast'][69:], rates_frame['nissSlow'][69:]))

    rates_frame['nissOsc'] = rates_frame['nissOscRaw'].rolling(window=1).mean()
    
    v = ema(rates_frame['nissOscRaw'], 24)
    rates_frame['nissSignal'] = [0]*(len(rates_frame['close']) - len(v)) + v
    rates_frame['sma'] = rates_frame['close'].rolling(window=200).mean()
    

    return rates_frame

In [48]:
def get_values(symbol, size):
    rates = mt5.copy_rates_from_pos(symbol, mt5.TIMEFRAME_H2, 0, size)

    rates_frame = pd.DataFrame(rates)
    
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume',], axis=1)
    rates_frame['sum'] = rates_frame['high'] + rates_frame['low']
    rates_frame['hl2'] = rates_frame['sum']/2 
    rates_frame['rsi'] = get_rsi(rates_frame['hl2'], 7)
    return rates_frame

In [49]:
symbol = "GBPUSD"
a = get_values(symbol, 5000)
a[-10:]

,open,high,low,close,sum,hl2,rsi
time,,,,,,,
2023-11-03 04:00:00,1.21909,1.22039,1.21903,1.21993,2.43942,1.219710,59.004033
2023-11-03 06:00:00,1.21993,1.22081,1.21971,1.22079,2.44052,1.220260,62.627749
2023-11-03 08:00:00,1.22080,1.22131,1.21964,1.21990,2.44095,1.220475,64.075924
2023-11-03 10:00:00,1.21990,1.22145,1.21855,1.22141,2.44000,1.220000,58.257253
2023-11-03 12:00:00,1.22141,1.22303,1.22111,1.22260,2.44414,1.222070,71.442175
2023-11-03 14:00:00,1.22260,1.23398,1.22244,1.23326,2.45642,1.228210,86.355902
2023-11-03 16:00:00,1.23327,1.23742,1.23106,1.23709,2.46848,1.234240,91.463647
2023-11-03 18:00:00,1.23708,1.23890,1.23576,1.23838,2.47466,1.237330,93.024751
2023-11-03 20:00:00,1.23838,1.23865,1.23713,1.23730,2.47578,1.237890,93.284420


In [64]:
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
b = a
B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
peck = 0
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []

for i in range(2,len(a)):
    if check == 1:
        sell_price = a.iloc[i].close
        pp = price_action(symbol, 1.0, buy_price, sell_price,mt5.ORDER_TYPE_SELL)
        print(f"{pp}---{a.iloc[i].close}--{a.iloc[i].name}")
        if pp<0.0:
            profit.append(-100.0)
        else:
            profit.append(pp)
        check = 0
    if a.iloc[i].rsi >= 90.0 and check == 0:
        buy_price = a.iloc[i].close #SELL Action
        print("#"*20)
        print(a.iloc[i].name)
        print(a.iloc[i].open)
        print("*"*20)
        check = 1
# for i in range(2,len(a)):
#     if check == 1:
#         sell_price = a.iloc[i].close
#         pp = price_action(symbol, 1.0, buy_price, sell_price,mt5.ORDER_TYPE_BUY)
#         print(f"{pp}---{a.iloc[i].close}--{a.iloc[i].name}")
#         profit.append(pp)
#         check = 0
#     if a.iloc[i].rsi <= 10.0 and check == 0:
#         buy_price = a.iloc[i].close #SELL Action
#         print("#"*20)
#         print(a.iloc[i].name)
#         print(a.iloc[i].open)
#         print("*"*20)
#         check = 1

#         if a.iloc[i].nissOsc > a.iloc[i].nissSignal:
#             print(f"{pp}---{a.iloc[i].close}--{a.iloc[i].name}")
#             check = 0
#             sell_price = a.iloc[i].close
#             pptest = price_action(symbol, 0.5, buy_price, sell_price,mt5.ORDER_TYPE_SELL)
#             if pp < 0.0:
#                 profit.append(pptest)
#             else:
#                 profit.append(pp)

####################
2007-10-05 16:00:00
2.0397
********************
180.0---2.0416--2007-10-05 18:00:00
####################
2007-10-05 18:00:00
2.043
********************
-40.0---2.042--2007-10-05 20:00:00
####################
2007-10-31 22:00:00
2.0795
********************
90.0---2.0793--2007-11-01 00:00:00
####################
2007-11-07 10:00:00
2.0964
********************
-30.0---2.1016--2007-11-07 12:00:00
####################
2007-11-07 12:00:00
2.1015
********************
-340.0---2.105--2007-11-07 14:00:00
####################
2007-11-07 14:00:00
2.1017
********************
90.0---2.1041--2007-11-07 16:00:00
####################
2007-11-07 16:00:00
2.1051
********************
-70.0---2.1048--2007-11-07 18:00:00
####################
2007-11-07 18:00:00
2.1039
********************
390.0---2.1009--2007-11-07 20:00:00
####################
2007-12-27 14:00:00
1.9910999999999999
********************
70.0---1.9907--2007-12-27 16:00:00
####################
2008-01-25 08:00:00
1.9781


####################
2010-03-30 14:00:00
1.50835
********************
49.0---1.50884--2010-03-30 16:00:00
####################
2010-03-30 16:00:00
1.50936
********************
235.0---1.5064899999999999--2010-03-30 18:00:00
####################
2010-06-10 20:00:00
1.46905
********************
139.0---1.4696799999999999--2010-06-10 22:00:00
####################
2010-06-10 22:00:00
1.47109
********************
-274.0---1.47242--2010-06-11 00:00:00
####################
2010-06-11 00:00:00
1.4696500000000001
********************
161.0---1.47081--2010-06-11 02:00:00
####################
2010-06-11 02:00:00
1.4723899999999999
********************
-27.0---1.47108--2010-06-11 04:00:00
####################
2010-07-14 12:00:00
1.5258099999999999
********************
-47.0---1.5239099999999999--2010-07-14 14:00:00
####################
2010-07-15 20:00:00
1.53791
********************
-6.0---1.54451--2010-07-15 22:00:00
####################
2010-07-15 22:00:00
1.54447
********************
224.0---1

####################
2013-11-25 00:00:00
1.62351
********************
27.0---1.6228--2013-11-25 02:00:00
####################
2013-12-26 16:00:00
1.64177
********************
72.0---1.64162--2013-12-26 18:00:00
####################
2013-12-27 08:00:00
1.64639
********************
-42.0---1.6491099999999999--2013-12-27 10:00:00
####################
2013-12-27 10:00:00
1.64869
********************
-472.0---1.6538300000000001--2013-12-27 12:00:00
####################
2013-12-27 12:00:00
1.64913
********************
123.0---1.6526--2013-12-27 14:00:00
####################
2014-02-12 16:00:00
1.65586
********************
-91.0---1.6585299999999998--2014-02-12 18:00:00
####################
2014-02-12 18:00:00
1.65761
********************
-64.0---1.65917--2014-02-12 20:00:00
####################
2014-02-12 20:00:00
1.65852
********************
-191.0---1.6610800000000001--2014-02-12 22:00:00
####################
2014-02-12 22:00:00
1.6591900000000002
********************
9.0---1.66099--2014-0

####################
2016-12-01 14:00:00
1.2622
********************
687.0---1.26213--2016-12-01 16:00:00
####################
2017-01-17 16:00:00
1.23327
********************
33.0---1.2382--2017-01-17 18:00:00
####################
2017-01-17 18:00:00
1.23854
********************
-130.0---1.2395--2017-01-17 20:00:00
####################
2017-01-17 20:00:00
1.23821
********************
-198.0---1.24148--2017-01-17 22:00:00
####################
2017-01-17 22:00:00
1.23949
********************
274.0---1.23874--2017-01-18 00:00:00
####################
2017-01-18 00:00:00
1.2412
********************
272.0---1.23602--2017-01-18 02:00:00
####################
2017-03-27 14:00:00
1.2585899999999999
********************
302.0---1.25698--2017-03-27 16:00:00
####################
2017-04-17 18:00:00
1.25934
********************
230.0---1.25629--2017-04-17 20:00:00
####################
2017-04-18 16:00:00
1.27189
********************
-17.0---1.2765900000000001--2017-04-18 18:00:00
##################

####################
2018-08-21 02:00:00
1.28007
********************
-91.0---1.28258--2018-08-21 04:00:00
####################
2018-08-21 04:00:00
1.28169
********************
-66.0---1.28324--2018-08-21 06:00:00
####################
2018-08-21 06:00:00
1.28257
********************
6.0---1.28318--2018-08-21 08:00:00
####################
2018-08-21 08:00:00
1.28323
********************
6.0---1.28312--2018-08-21 10:00:00
####################
2018-08-21 20:00:00
1.28787
********************
-8.0---1.29029--2018-08-21 22:00:00
####################
2018-08-21 22:00:00
1.29014
********************
-34.0---1.29063--2018-08-22 00:00:00
####################
2018-08-29 22:00:00
1.3020399999999999
********************
43.0---1.30232--2018-08-30 00:00:00
####################
2018-08-30 00:00:00
1.30218
********************
-76.0---1.30308--2018-08-30 02:00:00
####################
2018-08-30 02:00:00
1.30232
********************
8.0---1.303--2018-08-30 04:00:00
####################
2018-08-30 04:0

####################
2021-04-19 14:00:00
1.3918
********************
-231.0---1.39866--2021-04-19 16:00:00
####################
2021-04-19 16:00:00
1.39635
********************
8.0---1.39858--2021-04-19 18:00:00
####################
2021-04-19 18:00:00
1.39868
********************
-43.0---1.39901--2021-04-19 20:00:00
####################
2021-04-19 20:00:00
1.39858
********************
64.0---1.39837--2021-04-19 22:00:00
####################
2021-04-19 22:00:00
1.399
********************
-51.0---1.3988800000000001--2021-04-20 00:00:00
####################
2021-04-20 02:00:00
1.39889
********************
-117.0---1.39973--2021-04-20 04:00:00
####################
2021-05-07 18:00:00
1.39786
********************
-152.0---1.39968--2021-05-07 20:00:00
####################
2021-05-07 20:00:00
1.39817
********************
135.0---1.39833--2021-05-07 22:00:00
####################
2021-05-07 22:00:00
1.39968
********************
-458.0---1.4029099999999999--2021-05-10 00:00:00
#################

In [65]:
n = 0
p = 0
tn = 0
tp = 0

for i in profit:
    if i<0.0:
        n = n+i
        tn = tn+1
    else:
        p = p+i
        tp = tp+1
print(sum(profit))
print(f"Total negative sm -->{n}")
print(f"Total negative -->{tn}")      
print(f"Total positive sm -->{p}")      
print(f"Total positive -->{tp}") 
print(f"Length {len(profit)}")

19177.0
Total negative sm -->-24300.0
Total negative -->243
Total positive sm -->43477.0
Total positive -->279
Length 522
